In [3]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np

# 1. Definição da Mesma Arquitetura
class CNNBasica(nn.Module):
    def __init__(self):
        super(CNNBasica, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

classes = ('aviao', 'carro', 'passaro', 'gato', 'veado',
           'cao', 'sapo', 'cavalo', 'navio', 'caminhao')

# 2. Carregar o Modelo e Restaurar os Pesos
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
caminho_pesos = "modelo_cifar_cnn.pth"

if not os.path.exists(caminho_pesos):
    raise FileNotFoundError(f"Arquivo '{caminho_pesos}' nao encontrado. Execute o treinamento primeiro.")

modelo = CNNBasica().to(dispositivo)
# Carrega os pesos salvos e define modo de avaliacao
modelo.load_state_dict(torch.load(caminho_pesos, map_location=dispositivo))
modelo.eval()
print("Pesos do modelo carregados com sucesso!")

# 3. Carregar e Pre-processar a Imagem Externa (Ex.: bentoespock.jpeg)
caminho_imagem = "bentoespock.jpeg"
if not os.path.exists(caminho_imagem):
    caminho_imagem = "bentoespock_3.jpeg"

img_bgr = cv2.imread(caminho_imagem)
if img_bgr is None:
    raise FileNotFoundError(f"Imagem '{caminho_imagem}' nao encontrada!")

# A CNN treinada no CIFAR espera entradas exatamente 32x32 em RGB
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_redim = cv2.resize(img_rgb, (32, 32), interpolation=cv2.INTER_AREA)

# Normalizacao identica ao treinamento: (pixel / 255.0 - 0.5) / 0.5 -> [-1.0, 1.0]
tensor_img = img_redim.astype(np.float32) / 255.0
tensor_img = (tensor_img - 0.5) / 0.5

# Transposicao HWC -> CHW e adicao da dimensao do lote (Batch): (1, 3, 32, 32)
tensor_img = np.transpose(tensor_img, (2, 0, 1))
tensor_entrada = torch.tensor(tensor_img, dtype=torch.float32).unsqueeze(0).to(dispositivo)

# 4. Inferencia (Predicao)
with torch.no_grad():
    saida = modelo(tensor_entrada)
    # Aplica Softmax para transformar os logits brutos em probabilidades percentuais [0, 100%]
    probabilidades = torch.nn.functional.softmax(saida[0], dim=0)
    confianca, classe_id = torch.max(probabilidades, dim=0)

classe_predita = classes[classe_id.item()]
taxa_confianca = confianca.item() * 100

print(f"\nResultado da Inferencia:")
print(f"Classe Prevista: {classe_predita.upper()}")
print(f"Confianca: {taxa_confianca:.2f}%")

# 5. Exibicao Visual do Resultado
img_display = cv2.resize(img_bgr, (500, int(img_bgr.shape[0] * (500 / img_bgr.shape[1]))))
texto = f"Pred: {classe_predita} ({taxa_confianca:.1f}%)"
cv2.putText(img_display, texto, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

cv2.imshow("Teste do Modelo CNN", img_display)
cv2.waitKey(0)
cv2.destroyAllWindows()

Pesos do modelo carregados com sucesso!

Resultado da Inferencia:
Classe Prevista: GATO
Confianca: 41.41%
